## Skorch vs. TensorFlow: Comparación para Clasificación de Datos Tabulares
### 1. ¿Qué es Skorch?
Skorch es un wrapper que permite utilizar redes neuronales de PyTorch con una API similar a la de scikit-learn, lo que facilita la integración con pipelines y herramientas como GridSearchCV.


Se usa cuando queremos aprovechar la flexibilidad de PyTorch pero con la simplicidad de scikit-learn.
### 2. ¿Qué es TensorFlow?
TensorFlow es una librería de aprendizaje profundo desarrollada por Google, con herramientas como Keras, que simplifican la creación y ajuste de redes neuronales.


s más adecuado para modelos de producción, entrenamiento distribuido y optimización con aceleración por hardware (GPU/TPU).


### 3. Ajuste de Hiperparámetros
Para ambos frameworks, podemos ajustar:

* Arquitectura de la red: Número de capas, neuronas por capa.

* Funciones de activación: ReLU, Sigmoid, etc.

* Tasa de aprendizaje (learning_rate).

* Tamaño del batch (batch_size).

* Número de épocas (epochs).

* Optimizador: Adam, SGD, etc.


Para encontrar la mejor combinación, usaremos GridSearchCV con Skorch y Keras Tuner para TensorFlow.

In [ ]:
!pip install torch
!pip install skorch
!pip install keras-tuner

In [27]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split, GridSearchCV
from skorch import NeuralNetClassifier
from skorch.helper import SliceDataset
from skorch.callbacks import EarlyStopping
from skorch.callbacks import EpochScoring


In [6]:
# Carga del dataset de simulaciones
data=pd.read_csv('./data/C2018-13-Christmas-market/simulation_C2018-13-Christmas-market_tp2_ev0.0713.csv',index_col=0)
data.head(2)

,duration1,duration2,duration3,duration4,duration5,duration6,duration7,duration8,duration21,duration9,...,duration26,duration28,duration29,duration30,duration31,duration32,duration33,critical_path,baseline_duration,actual_duration
0,0,59,257.956695,10.093668,18.189347,266.724319,112.345085,128.122007,46.119746,179.073063,...,17.321310,9.933554,11.313659,10.880869,16.825025,7.786062,0,"[1, 2, 3, 4, 6, 13, 27, 28, 29, 31, 32, 33]",1107.6,1128.504156
1,0,59,277.539098,10.225146,16.415950,273.195967,115.535606,121.148158,48.234513,134.272704,...,16.494392,10.049567,10.605734,10.358704,18.822929,9.487006,0,"[1, 2, 3, 4, 6, 13, 27, 28, 29, 31, 32, 33]",1107.6,1186.873800


In [7]:
# Creo la columna delay y elimino baseline duration y actual duration
data['delay']=(data['actual_duration']>data['baseline_duration'])*1
data.drop(columns=['baseline_duration','actual_duration','critical_path'],inplace=True)
data.head(2)

,duration1,duration2,duration3,duration4,duration5,duration6,duration7,duration8,duration21,duration9,...,duration23,duration25,duration26,duration28,duration29,duration30,duration31,duration32,duration33,delay
0,0,59,257.956695,10.093668,18.189347,266.724319,112.345085,128.122007,46.119746,179.073063,...,8.301023,8.483037,17.321310,9.933554,11.313659,10.880869,16.825025,7.786062,0,1
1,0,59,277.539098,10.225146,16.415950,273.195967,115.535606,121.148158,48.234513,134.272704,...,7.211210,8.247698,16.494392,10.049567,10.605734,10.358704,18.822929,9.487006,0,1


In [9]:
# Split the dataset into features and class
X = data.drop(columns=['delay'])
y = data['delay']

# Se usa stratify=y en train_test_split para asegurar que la distribución de clases sea la misma en entrenamiento y prueba.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Verificar la distribución de clases (me están saliendo accuracies de 1 xq sólo tenemos datos con retraso)
print("Unique labels in training:", np.unique(y_train, return_counts=True))
print("Unique labels in test:", np.unique(y_test, return_counts=True))

Unique labels in training: (array([0, 1]), array([212, 588]))
Unique labels in test: (array([0, 1]), array([ 53, 147]))


In [37]:
# ---- Modelo con Skorch ----
class SkorchNet(nn.Module):
    def __init__(self, input_dim=33, hidden_dim=64, output_dim=2):
        super(SkorchNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)  # Normalización (como hace TF)
        self.relu = nn.ReLU()
        #self.dropout = nn.Dropout(0.5)  # Agrega Dropout
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, X):
        X = self.fc1(X)
        X = self.relu(X)
        #X = self.dropout(X)  # Dropout aplicado aquí
        X = self.fc2(X)
        return self.softmax(X)

net = NeuralNetClassifier(
    SkorchNet,
    module__input_dim=33,
    module__hidden_dim=64,  # Valor por defecto, pero se tuneará en GridSearch
    module__output_dim=2,
    max_epochs=30,
    lr=0.001,
    optimizer=torch.optim.AdamW,
    optimizer__weight_decay=0.01,  # Regularización L2 penaliza los pesos grandes en la red neuronal para evitar el sobreajuste (overfitting)
    criterion=nn.CrossEntropyLoss,
    batch_size=32,
    iterator_train__shuffle=True,
    callbacks=[EarlyStopping(patience=5, load_best=True), EpochScoring('accuracy', on_train=True), EpochScoring('accuracy', on_train=False)]


)

# GridSearch para Skorch, ahora tunnea hidden_dim
params = {
    'lr': [0.01, 0.001, 0.0001, 0.00005],
    'batch_size': [8, 16, 32, 64],
    'module__hidden_dim': [32, 64, 128, 256]  # Ahora se ajusta hidden_dim
}

gs = GridSearchCV(net, params, scoring='accuracy', cv=5)
gs.fit(X_train.values.astype(np.float32), y_train.values)

print("Best Skorch Model:", gs.best_params_)
print("Best Skorch Accuracy:", gs.best_score_)


# Como detectar overfitting
#Si train_loss baja pero valid_loss sube, el modelo está memorizando los datos de entrenamiento y no generaliza bien.
#Si train_acc es mucho mayor que valid_acc, también es señal de sobreajuste.

  epoch    accuracy    train_loss    valid_acc    valid_loss     dur
-------  ----------  ------------  -----------  ------------  ------
      1      0.7344        0.5828       0.7344        0.5789  0.0349
      2      0.7344        0.5789       0.7344        0.5789  0.0292
      3      0.7344        0.5789       0.7344        0.5789  0.0264
      4      0.7344        0.5789       0.7344        0.5789  0.0250
      5      0.7344        0.5789       0.7344        0.5789  0.0248
Stopping since valid_loss has not improved in the last 5 epochs.
Restoring best model from epoch 1.
  epoch    accuracy    train_loss    valid_acc    valid_loss     dur
-------  ----------  ------------  -----------  ------------  ------
      1      0.2656        1.0476       0.2656        1.0476  0.0251
      2      0.2656        1.0476       0.2656        1.0476  0.0256
      3      0.2656        1.0476       0.2656        1.0476  0.0654
      4      0.2656        1.0476       0.2656        1.0476  0.0253
   

In [38]:
# ---- Modelo con TensorFlow ----
def build_model(hp):
    model = keras.Sequential([
        layers.Dense(hp.Int('units', min_value=32, max_value=128, step=32), activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        #layers.Dropout(0.5),  # Agrega Dropout
        layers.Dense(2, activation='softmax')
    ])
    model.compile(optimizer=keras.optimizers.Adam(hp.Choice('learning_rate', [0.01, 0.001, 0.0001])),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

from kerastuner.tuners import RandomSearch
tuner = RandomSearch(build_model, objective='val_accuracy', max_trials=10, executions_per_trial=1, directory='tuner_dir')

tuner.search(X_train, y_train, epochs=30, validation_split=0.2, batch_size=32)
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"Best TensorFlow Model: units={best_hps.get('units')}, learning_rate={best_hps.get('learning_rate')}")

# Evaluación final con los mejores hiperparámetros
best_model = tuner.hypermodel.build(best_hps)
best_model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.2, callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
test_loss, test_acc = best_model.evaluate(X_test, y_test)

print("Best TensorFlow Accuracy:", test_acc)


Reloading Tuner from tuner_dir/untitled_project/tuner0.json
Best TensorFlow Model: units=64, learning_rate=0.001
Epoch 1/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6506 - loss: 4.1890 - val_accuracy: 0.4688 - val_loss: 4.8741
Epoch 2/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5709 - loss: 4.2019 - val_accuracy: 0.6438 - val_loss: 2.8495
Epoch 3/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6027 - loss: 3.2529 - val_accuracy: 0.5562 - val_loss: 2.4883
Epoch 4/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6398 - loss: 2.2886 - val_accuracy: 0.7188 - val_loss: 2.7145
Epoch 5/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6828 - loss: 2.5324 - val_accuracy: 0.5750 - val_loss: 2.7472
Epoch 6/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6202 - loss: 2.5750 - val_accuracy: 0.5875 - val_loss: 2.0858
Epoch 7/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6843 - loss: 1.8076 - val_accuracy: 0.7125 - val_loss: 1.5491
Epoch